# FVC endmember calibration from Earth Engine

This diagnostic notebook recalculates the statistical NDVI endmembers used for fractional vegetation cover (FVC) **from the satellite products and repository code**.

It does **not** depend on previous CSV or Excel diagnostic files.

The notebook remains outside the production pipeline. Its required production output is:

`config/fvc_endmembers.json`

The production pipeline should fail explicitly if this calibration file is absent or invalid.

## Fixed calibration method

For each optical source (`HLS` and `S2`) independently:

1. Use the MODIS 8-day periods from the study interval.
2. Build the source-specific optical medoid for every period using the current repository modules.
3. Define optical coverage from the common valid mask of `Green`, `Red`, and `NIR`.
4. Retain station-period observations with `coverage_pct >= 80`.
5. Calculate NDVI and NDWI from the medoid.
6. Exclude water pixels using `NDWI > 0`.
7. Within each valid station-period:
   - low candidate = NDVI P05
   - high candidate = NDVI P95
8. Across all valid 2021–2023 station-period observations:
   - statistical low-NDVI endmember = P05 of all station-period P05 values
   - statistical high-NDVI endmember = P95 of all station-period P95 values

The endmembers are global in time and source-specific.


In [21]:
from __future__ import annotations

import json
import sys
import time
from datetime import date, timedelta
from pathlib import Path

import ee
import pandas as pd


## 1. Repository setup and Earth Engine initialization

The notebook searches upward for the repository root, so it can be executed with a clean kernel from `notebooks/diagnostics/`.


In [22]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found. Expected to find pyproject.toml."
    )


REPO_ROOT = find_repo_root()
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Repository:", REPO_ROOT)
print("Source path:", SRC_PATH)


Repository: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion
Source path: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\src


In [23]:
EE_PROJECT = "ee-change"

ee.Initialize(project=EE_PROJECT)
ee.Number(1).getInfo()

print("Earth Engine initialized with project:", EE_PROJECT)


Earth Engine initialized with project: ee-change


## 2. Import the current repository processing functions

The calibration deliberately reuses the current optical preprocessing and medoid functions instead of reimplementing sensor QA in the notebook.


In [24]:
from et_downscaling.hls import (
    build_hls_medoid,
    get_hls_collection,
)
from et_downscaling.modis import build_modis_inputs
from et_downscaling.sentinel2 import (
    build_s2_medoid,
    get_sentinel2_collection,
)


## 3. Calibration settings

`outputs/diagnostics/` is used only for ignored diagnostic checkpoints and QA tables.

`config/fvc_endmembers.json` is the versioned calibration contract consumed by the production pipeline.


In [25]:
COVERAGE_THRESHOLD_PCT = 80.0
SOURCE_SCALES_M = {
    "HLS": 30,
    "S2": 20,
}

WITHIN_PERIOD_LOW_PERCENTILE = 5
WITHIN_PERIOD_HIGH_PERCENTILE = 95
GLOBAL_LOW_QUANTILE = 0.05
GLOBAL_HIGH_QUANTILE = 0.95

# Set True to ignore this notebook's own checkpoint and recompute every period.
FORCE_REBUILD = False

DIAGNOSTIC_DIR = REPO_ROOT / "outputs" / "diagnostics"
CHECKPOINT_PATH = (
    DIAGNOSTIC_DIR / "_fvc_endmember_calibration_checkpoint.csv"
)
OBSERVATION_OUTPUT_PATH = (
    DIAGNOSTIC_DIR / "fvc_endmember_calibration_observations.csv"
)
ENDMEMBER_TABLE_PATH = (
    DIAGNOSTIC_DIR / "fvc_global_endmembers.csv"
)
PIPELINE_CONFIG_PATH = (
    REPO_ROOT / "config" / "fvc_endmembers.json"
)

DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Checkpoint:", CHECKPOINT_PATH)
print("Pipeline calibration:", PIPELINE_CONFIG_PATH)


Checkpoint: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\outputs\diagnostics\_fvc_endmember_calibration_checkpoint.csv
Pipeline calibration: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\config\fvc_endmembers.json


## 4. Build station footprints and optical collections from source data

No previous diagnostic table is loaded here.


In [26]:
modis_inputs = build_modis_inputs()

modis_collection = ee.ImageCollection(modis_inputs["collection"])
station_footprints = ee.FeatureCollection(
    modis_inputs["station_footprints"]
)

s2_collection = get_sentinel2_collection(station_footprints)
hls_collection = get_hls_collection(station_footprints)

station_count = int(station_footprints.size().getInfo())

station_properties = station_footprints.select(
    [
        "station",
        "station_id",
        "longitude",
        "latitude",
        "footprint_area_m2",
    ]
).getInfo()["features"]

stations = [
    feature["properties"]
    for feature in station_properties
]

print("Stations:", station_count)
print([station["station"] for station in stations])


Stations: 5
['Pastos limpios', 'Palma', 'Bananera', 'Manglar', 'Bosque seco']


## 5. Get the MODIS 8-day calibration periods

The optical endmembers are calibrated on the same temporal support used by the ET target.


In [27]:
period_millis = (
    modis_collection
    .aggregate_array("system:time_start")
    .getInfo()
)

period_starts = sorted(
    {
        pd.to_datetime(
            value,
            unit="ms",
            utc=True,
        ).date()
        for value in period_millis
    }
)


def get_period_end(period_start: date) -> date:
    regular_end = period_start + timedelta(days=8)
    next_year_start = date(period_start.year + 1, 1, 1)
    return min(regular_end, next_year_start)


periods = [
    (period_start, get_period_end(period_start))
    for period_start in period_starts
]

print("MODIS periods:", len(periods))
print("First:", periods[0])
print("Last:", periods[-1])


MODIS periods: 138
First: (datetime.date(2021, 1, 1), datetime.date(2021, 1, 9))
Last: (datetime.date(2023, 12, 27), datetime.date(2024, 1, 1))


## 6. Earth Engine helper functions

Coverage is computed before the water mask from the common valid support of `Green`, `Red`, and `NIR`.

The water mask is applied only to the NDVI distribution used for endmember candidates.


In [28]:
def safe_normalized_difference(
    image: ee.Image,
    positive_band: str,
    negative_band: str,
    output_name: str,
    epsilon: float = 1e-6,
) -> ee.Image:
    image = ee.Image(image)

    positive = image.select(positive_band)
    negative = image.select(negative_band)

    numerator = positive.subtract(negative)
    denominator = positive.add(negative)

    return (
        numerator
        .divide(denominator)
        .updateMask(denominator.abs().gt(epsilon))
        .rename(output_name)
        .toFloat()
    )


def build_ndvi_ndwi(medoid: ee.Image) -> ee.Image:
    medoid = ee.Image(medoid)

    ndvi = safe_normalized_difference(
        medoid,
        "NIR",
        "Red",
        "NDVI",
    )

    ndwi = safe_normalized_difference(
        medoid,
        "Green",
        "NIR",
        "NDWI",
    )

    return ndvi.addBands(ndwi)


def get_common_valid_mask(medoid: ee.Image) -> ee.Image:
    return (
        ee.Image(medoid)
        .select(["Green", "Red", "NIR"])
        .mask()
        .reduce(ee.Reducer.min())
        .rename("valid")
    )


def get_info_with_retry(
    ee_object,
    attempts: int = 5,
    base_sleep_seconds: int = 5,
):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            return ee_object.getInfo()

        except Exception as error:
            last_error = error

            if attempt == attempts:
                break

            sleep_seconds = base_sleep_seconds * attempt
            print(
                f"Earth Engine request failed "
                f"(attempt {attempt}/{attempts}). "
                f"Retrying in {sleep_seconds}s..."
            )
            time.sleep(sleep_seconds)

    raise last_error


## 7. Calculate one source-period group

For every period, the medoid is built once for the union of the five station footprints. Statistics are then evaluated independently for each footprint.

Low-coverage observations remain in the diagnostic table, but their NDVI endmember candidates are not used.


In [29]:
def build_source_period_rows(
    source: str,
    period_start: date,
    period_end: date,
) -> list[dict]:
    if source == "HLS":
        full_collection = hls_collection
        build_medoid = build_hls_medoid

    elif source == "S2":
        full_collection = s2_collection
        build_medoid = build_s2_medoid

    else:
        raise ValueError(f"Unsupported optical source: {source}")

    scale = SOURCE_SCALES_M[source]
    start_text = period_start.isoformat()
    end_text = period_end.isoformat()

    period_collection = (
        ee.ImageCollection(full_collection)
        .filterDate(start_text, end_text)
        .filterBounds(station_footprints.geometry())
    )

    product_count = int(
        get_info_with_retry(period_collection.size())
    )

    # No optical products: record five explicit zero-coverage observations.
    if product_count == 0:
        rows = []

        for station in stations:
            rows.append(
                {
                    "source": source,
                    "station": station.get("station"),
                    "station_id": station.get("station_id"),
                    "period_start": start_text,
                    "period_end": end_text,
                    "source_scale_m": scale,
                    "products": 0,
                    "coverage_pct": 0.0,
                    "NDVI_p05": None,
                    "NDVI_p95": None,
                    "nonwater_pixel_count": 0,
                }
            )

        return rows

    analysis_geometry = station_footprints.geometry()

    medoid = ee.Image(
        build_medoid(
            period_collection,
            analysis_geometry,
        )
    )

    indices = build_ndvi_ndwi(medoid)

    valid_mask = get_common_valid_mask(medoid)
    ndvi = indices.select("NDVI")
    ndwi = indices.select("NDWI")

    nonwater_ndvi = ndvi.updateMask(
        ndwi.lte(0)
    )

    projection = medoid.select("NIR").projection()

    percentile_reducer = (
        ee.Reducer.percentile(
            [
                WITHIN_PERIOD_LOW_PERCENTILE,
                WITHIN_PERIOD_HIGH_PERCENTILE,
            ]
        )
        .combine(
            reducer2=ee.Reducer.count(),
            sharedInputs=True,
        )
    )

    def add_statistics(feature):
        feature = ee.Feature(feature)
        geometry = feature.geometry()

        coverage_raw = (
            valid_mask
            .unmask(0)
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=geometry,
                crs=projection,
                scale=scale,
                maxPixels=1e6,
                tileScale=4,
            )
            .get("valid")
        )

        coverage_pct = ee.Number(
            ee.Algorithms.If(
                ee.Algorithms.IsEqual(
                    coverage_raw,
                    None,
                ),
                0,
                ee.Number(coverage_raw).multiply(100),
            )
        )

        raw_ndvi_statistics = ee.Dictionary(
            ee.Algorithms.If(
                coverage_pct.gte(
                    COVERAGE_THRESHOLD_PCT
                ),
                nonwater_ndvi.reduceRegion(
                    reducer=percentile_reducer,
                    geometry=geometry,
                    crs=projection,
                    scale=scale,
                    maxPixels=1e6,
                    tileScale=4,
                ),
                ee.Dictionary({}),
            )
        )

        # Some high-coverage footprints can contain zero non-water pixels.
        # In that case Earth Engine omits percentile keys entirely.
        # Fill missing keys with explicit sentinels so the mapped
        # FeatureCollection can be evaluated without failing.
        ndvi_statistics = ee.Dictionary(
            {
                "p5": -9999,
                "p95": -9999,
                "count": 0,
            }
        ).combine(
            raw_ndvi_statistics,
            overwrite=True,
        )

        return feature.set(
            {
                "source": source,
                "period_start": start_text,
                "period_end": end_text,
                "source_scale_m": scale,
                "products": product_count,
                "coverage_pct": coverage_pct,
                "NDVI_p05": ndvi_statistics.get("p5"),
                "NDVI_p95": ndvi_statistics.get("p95"),
                "nonwater_pixel_count": ndvi_statistics.get(
                    "count"
                ),
            }
        )

    result = ee.FeatureCollection(
        station_footprints.map(add_statistics)
    )

    info = get_info_with_retry(result)

    rows = []

    for feature in info["features"]:
        properties = feature["properties"]

        rows.append(
            {
                "source": properties.get("source"),
                "station": properties.get("station"),
                "station_id": properties.get("station_id"),
                "period_start": properties.get("period_start"),
                "period_end": properties.get("period_end"),
                "source_scale_m": properties.get(
                    "source_scale_m"
                ),
                "products": properties.get("products"),
                "coverage_pct": properties.get(
                    "coverage_pct"
                ),
                "NDVI_p05": properties.get("NDVI_p05"),
                "NDVI_p95": properties.get("NDVI_p95"),
                "nonwater_pixel_count": properties.get(
                    "nonwater_pixel_count"
                ),
            }
        )

    return rows


## 8. Recalculate all HLS and Sentinel-2 station-period observations

This cell is restartable. Its checkpoint is generated by this notebook and lives under `outputs/`, which can remain ignored by Git.

With no checkpoint present, the notebook starts from the satellite products and recalculates everything.


In [30]:
checkpoint_columns = [
    "source",
    "station",
    "station_id",
    "period_start",
    "period_end",
    "source_scale_m",
    "products",
    "coverage_pct",
    "NDVI_p05",
    "NDVI_p95",
    "nonwater_pixel_count",
]

if FORCE_REBUILD and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()

if CHECKPOINT_PATH.exists():
    observations = pd.read_csv(CHECKPOINT_PATH)
    print(
        "Loaded notebook checkpoint:",
        len(observations),
        "rows",
    )
else:
    observations = pd.DataFrame(
        columns=checkpoint_columns
    )
    print("No checkpoint found. Starting from source data.")


def group_is_complete(
    table: pd.DataFrame,
    source: str,
    period_start: str,
) -> bool:
    subset = table.loc[
        (table["source"] == source)
        & (table["period_start"] == period_start)
    ]

    return subset["station_id"].nunique() == station_count


total_groups = len(periods) * len(SOURCE_SCALES_M)
group_index = 0

for source in SOURCE_SCALES_M:
    for period_start, period_end in periods:
        group_index += 1
        start_text = period_start.isoformat()

        if group_is_complete(
            observations,
            source,
            start_text,
        ):
            print(
                f"{group_index}/{total_groups} | "
                f"{source} | {start_text} | cached"
            )
            continue

        print(
            f"{group_index}/{total_groups} | "
            f"{source} | "
            f"{start_text} -> {period_end.isoformat()}"
        )

        rows = build_source_period_rows(
            source,
            period_start,
            period_end,
        )

        # Replace a partial group if a previous run stopped mid-write.
        if not observations.empty:
            observations = observations.loc[
                ~(
                    (observations["source"] == source)
                    & (
                        observations["period_start"]
                        == start_text
                    )
                )
            ].copy()

        observations = pd.concat(
            [
                observations,
                pd.DataFrame(rows),
            ],
            ignore_index=True,
        )

        observations = observations[
            checkpoint_columns
        ].sort_values(
            [
                "source",
                "period_start",
                "station_id",
            ]
        )

        observations.to_csv(
            CHECKPOINT_PATH,
            index=False,
        )

print("Final station-period rows:", len(observations))


Loaded notebook checkpoint: 5 rows
1/276 | HLS | 2021-01-01 | cached
2/276 | HLS | 2021-01-09 -> 2021-01-17


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


3/276 | HLS | 2021-01-17 -> 2021-01-25


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


4/276 | HLS | 2021-01-25 -> 2021-02-02


...

275/276 | S2 | 2023-12-19 -> 2023-12-27


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


276/276 | S2 | 2023-12-27 -> 2024-01-01


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Final station-period rows: 1380


## 9. Quality control and eligibility

Expected complete design:

- 138 MODIS periods
- 5 station footprints
- 2 optical sources
- 1,380 source × station × period rows

The historical diagnostic run produced 434 eligible HLS observations and 498 eligible Sentinel-2 observations at the 80% threshold. The counts below are recalculated, not imported.


In [31]:
expected_rows = (
    len(periods)
    * station_count
    * len(SOURCE_SCALES_M)
)

if len(observations) != expected_rows:
    raise ValueError(
        f"Incomplete calibration table: "
        f"{len(observations)} rows found, "
        f"{expected_rows} expected."
    )

observations["coverage_pct"] = pd.to_numeric(
    observations["coverage_pct"],
    errors="coerce",
)
observations["NDVI_p05"] = pd.to_numeric(
    observations["NDVI_p05"],
    errors="coerce",
)
observations["NDVI_p95"] = pd.to_numeric(
    observations["NDVI_p95"],
    errors="coerce",
)
observations["nonwater_pixel_count"] = pd.to_numeric(
    observations["nonwater_pixel_count"],
    errors="coerce",
)

# Convert explicit Earth Engine missing-value sentinels to NaN.
for column in ["NDVI_p05", "NDVI_p95"]:
    observations.loc[
        observations[column] <= -9990,
        column,
    ] = pd.NA

eligible = observations.loc[
    (observations["coverage_pct"] >= COVERAGE_THRESHOLD_PCT)
    & observations["NDVI_p05"].notna()
    & observations["NDVI_p95"].notna()
    & (observations["nonwater_pixel_count"] > 0)
].copy()

coverage_summary = (
    observations.groupby("source")
    .agg(
        total_observations=("source", "size"),
        mean_coverage_pct=("coverage_pct", "mean"),
        median_coverage_pct=("coverage_pct", "median"),
        eligible_observations=(
            "coverage_pct",
            lambda values: (
                values >= COVERAGE_THRESHOLD_PCT
            ).sum(),
        ),
    )
    .reset_index()
)

eligibility_summary = (
    eligible.groupby("source")
    .agg(
        eligible_observations=("source", "size"),
        stations=("station", "nunique"),
        ndvi_p05_min=("NDVI_p05", "min"),
        ndvi_p05_median=("NDVI_p05", "median"),
        ndvi_p05_max=("NDVI_p05", "max"),
        ndvi_p95_min=("NDVI_p95", "min"),
        ndvi_p95_median=("NDVI_p95", "median"),
        ndvi_p95_max=("NDVI_p95", "max"),
    )
    .reset_index()
)

display(coverage_summary.round(4))
display(eligibility_summary.round(6))


,source,total_observations,mean_coverage_pct,median_coverage_pct,eligible_observations
0,HLS,690,67.2431,100.0,434
1,S2,690,76.1477,100.0,498


,source,eligible_observations,stations,ndvi_p05_min,ndvi_p05_median,ndvi_p05_max,ndvi_p95_min,ndvi_p95_median,ndvi_p95_max
0,HLS,5,5,0.510267,0.661234,0.751651,0.827366,0.876207,0.908302


## 10. Calculate the global source-specific endmembers

Final calibration rule:

- `ndvi_low_endmember = P05(all eligible NDVI_p05)`
- `ndvi_high_endmember = P95(all eligible NDVI_p95)`


In [32]:
endmembers = (
    eligible.groupby("source")
    .agg(
        ndvi_low_endmember=(
            "NDVI_p05",
            lambda values: values.quantile(
                GLOBAL_LOW_QUANTILE
            ),
        ),
        ndvi_high_endmember=(
            "NDVI_p95",
            lambda values: values.quantile(
                GLOBAL_HIGH_QUANTILE
            ),
        ),
        n_observations=("source", "size"),
        n_stations=("station", "nunique"),
        period_start=("period_start", "min"),
        period_end=("period_end", "max"),
    )
    .reset_index()
)

endmembers["ndvi_range"] = (
    endmembers["ndvi_high_endmember"]
    - endmembers["ndvi_low_endmember"]
)

if set(endmembers["source"]) != {"HLS", "S2"}:
    raise ValueError(
        "Calibration must contain both HLS and S2."
    )

if not (endmembers["ndvi_range"] > 0).all():
    raise ValueError(
        "NDVI endmember range must be positive."
    )

if not (
    endmembers["ndvi_low_endmember"]
    .between(-1, 1)
    .all()
    and endmembers["ndvi_high_endmember"]
    .between(-1, 1)
    .all()
):
    raise ValueError(
        "NDVI endmembers must be within [-1, 1]."
    )

display(endmembers.round(6))


ValueError: Calibration must contain both HLS and S2.

## 11. Compare with the previously observed diagnostic values

These reference values are used **only as a reproducibility check**. They are not used to calculate or overwrite the new calibration.

Small differences can occur if repository preprocessing or Earth Engine source data change. Large differences should trigger methodological review before updating the production calibration.


In [ ]:
reference_values = pd.DataFrame(
    [
        {
            "source": "HLS",
            "reference_low": 0.416392,
            "reference_high": 0.908363,
        },
        {
            "source": "S2",
            "reference_low": 0.309061,
            "reference_high": 0.924045,
        },
    ]
)

reproducibility_check = endmembers.merge(
    reference_values,
    on="source",
    how="left",
)

reproducibility_check["low_difference"] = (
    reproducibility_check["ndvi_low_endmember"]
    - reproducibility_check["reference_low"]
)

reproducibility_check["high_difference"] = (
    reproducibility_check["ndvi_high_endmember"]
    - reproducibility_check["reference_high"]
)

display(
    reproducibility_check[
        [
            "source",
            "ndvi_low_endmember",
            "reference_low",
            "low_difference",
            "ndvi_high_endmember",
            "reference_high",
            "high_difference",
        ]
    ].round(6)
)


## 12. Save diagnostic outputs and the production calibration contract

The diagnostic CSV files remain under `outputs/` and may be ignored by Git.

The only file required by the production pipeline is:

`config/fvc_endmembers.json`

That JSON is generated from the values recalculated in this notebook.


In [ ]:
observations.to_csv(
    OBSERVATION_OUTPUT_PATH,
    index=False,
)

endmembers.to_csv(
    ENDMEMBER_TABLE_PATH,
    index=False,
)

source_config = {}

for row in endmembers.itertuples(index=False):
    source_config[row.source] = {
        "ndvi_low_endmember": float(
            row.ndvi_low_endmember
        ),
        "ndvi_high_endmember": float(
            row.ndvi_high_endmember
        ),
        "n_observations": int(
            row.n_observations
        ),
        "n_stations": int(
            row.n_stations
        ),
    }

calibration = {
    "calibration_name": (
        "fvc_global_ndvi_endmembers"
    ),
    "calibration_scope": (
        "external_diagnostic_prerequisite"
    ),
    "method": (
        "two_stage_global_percentile"
    ),
    "coverage_threshold_pct": (
        COVERAGE_THRESHOLD_PCT
    ),
    "coverage_bands": [
        "Green",
        "Red",
        "NIR",
    ],
    "water_exclusion": {
        "index": "NDWI",
        "rule": "exclude NDWI > 0",
    },
    "within_footprint_period": {
        "low_candidate_percentile": (
            WITHIN_PERIOD_LOW_PERCENTILE
        ),
        "high_candidate_percentile": (
            WITHIN_PERIOD_HIGH_PERCENTILE
        ),
    },
    "across_all_valid_observations": {
        "low_quantile": GLOBAL_LOW_QUANTILE,
        "high_quantile": GLOBAL_HIGH_QUANTILE,
        "temporal_scheme": "global",
        "sources_calibrated_separately": True,
    },
    "calibration_period": {
        "start": min(period_starts).isoformat(),
        "end": max(
            period_end
            for _, period_end in periods
        ).isoformat(),
    },
    "sources": source_config,
}

with PIPELINE_CONFIG_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        calibration,
        file,
        indent=2,
        ensure_ascii=False,
    )
    file.write("\n")

print(
    "Saved observations:",
    OBSERVATION_OUTPUT_PATH,
)
print(
    "Saved endmember table:",
    ENDMEMBER_TABLE_PATH,
)
print(
    "Saved pipeline calibration:",
    PIPELINE_CONFIG_PATH,
)

print("\nFinal calibration:")
for source, values in calibration["sources"].items():
    print(
        f"{source}: "
        f"low={values['ndvi_low_endmember']:.6f}, "
        f"high={values['ndvi_high_endmember']:.6f}, "
        f"n={values['n_observations']}"
    )


## Production pipeline requirement

This notebook is **not** part of the operational ET/FVC pipeline.

The production pipeline must treat `config/fvc_endmembers.json` as a prerequisite and should:

1. fail if the file does not exist;
2. fail if the requested optical source is absent;
3. validate `-1 <= low < high <= 1`;
4. use the stored source-specific endmembers;
5. never recalculate endmembers silently during production execution.

If the calibration methodology changes, this notebook should be rerun explicitly and the updated JSON reviewed and versioned.
